# NumPy

In [1]:
import numpy as np

## Memory

The actual data of a numpy array is stored in a homogeneous and contiguous block of memory called data buffer. 

![Memory](./Images/PythonMemory.png)

### Strides
Stride is a tuple of bytes to step in each dimension when traversing an array.

#### Strides in 1D

Memory is just one long row of bytes. Let's create 5 numbers of type `int32` – each takes **4 bytes** (we use `int32` on purpose, so that "4 bytes" and "1 element" are easy to tell apart):

```
element:       10      20      30      40      50
index:          0       1       2       3       4
byte offset:    0       4       8      12      16
            |-------|-------|-------|-------|-------|
```

To get from one element to the next, NumPy has to jump **4 bytes**. That number is the stride.

In [2]:
x = np.array([10, 20, 30, 40, 50], dtype=np.int32)

print("x:", x)
print("itemsize:", x.itemsize, "bytes")
print("strides: ", x.strides)   # (4,) -> next element is 4 bytes further

x: [10 20 30 40 50]
itemsize: 4 bytes
strides:  (4,)


The address of element `x[i]` is therefore:

```
offset(x[i]) = i * strides[0]
```

For example `x[3]` starts at byte `3 * 4 = 12`. Let's check it for every element:

In [3]:
for i in range(len(x)):
    offset = i * x.strides[0]
    print(f"x[{i}] = {x[i]}  ->  starts at byte {offset}")

x[0] = 10  ->  starts at byte 0
x[1] = 20  ->  starts at byte 4
x[2] = 30  ->  starts at byte 8
x[3] = 40  ->  starts at byte 12
x[4] = 50  ->  starts at byte 16


#### Slicing only changes the strides

A slice with a step does **not copy** anything. NumPy uses the same memory and only changes the stride:

* `x[::2]` – take every second element → jump **2 × 4 = 8 bytes**
* `x[::-1]` – go backwards → jump **−4 bytes**

```
x[::2]:        10              30              50
            |-------|-------|-------|-------|-------|
                ^-------8------>^-------8------>^
```

In [4]:
every_second = x[::2]
reversed_x = x[::-1]

print("x[::2]: ", every_second, " strides:", every_second.strides)
print("x[::-1]:", reversed_x, " strides:", reversed_x.strides)
print("Same memory as x?", np.shares_memory(x, every_second), np.shares_memory(x, reversed_x))

x[::2]:  [10 30 50]  strides: (8,)
x[::-1]: [50 40 30 20 10]  strides: (-4,)
Same memory as x? True True


#### Strides in 2D

A 2D array is still stored in memory as **one row of bytes**. Rows are simply placed one after another (this is the *C order*, `C_CONTIGUOUS`):

```
m = [[10, 20, 30],
     [40, 50, 60]]

memory:       10  20  30 | 40  50  60
byte offset:   0   4   8 | 12  16  20
              \_ row 0 _/  \_ row 1 _/
```

Now we need **two** strides – one for each axis:

* **axis 1** (next column, same row): jump 1 element = **4 bytes**
* **axis 0** (next row, same column): jump a whole row = 3 elements × 4 bytes = **12 bytes**

So `m.strides == (12, 4)`.

In [5]:
m = np.array([[10, 20, 30],
              [40, 50, 60]], dtype=np.int32)

print("m:\n", m)
print("shape:   ", m.shape)
print("itemsize:", m.itemsize)
print("strides: ", m.strides)   # (12, 4)

m:
 [[10 20 30]
 [40 50 60]]
shape:    (2, 3)
itemsize: 4
strides:  (12, 4)


The address of element `m[i, j]` is:

```
offset(m[i, j]) = i * strides[0] + j * strides[1]
                = i * 12         + j * 4
```

For example `m[1, 2]`: `1 * 12 + 2 * 4 = 20` bytes → that is element number `20 / 4 = 5` in memory, i.e. the value `60`.

Let's check this for all elements. `m.ravel()` shows the memory as one row, so we can look up the element by its position there:

In [6]:
memory = m.ravel()   # the data buffer seen as one row

for i in range(m.shape[0]):
    for j in range(m.shape[1]):
        offset = i * m.strides[0] + j * m.strides[1]
        position = offset // m.itemsize
        print(f"m[{i}, {j}] = {m[i, j]}  ->  byte {offset:2d}  ->  memory[{position}] = {memory[position]}")

m[0, 0] = 10  ->  byte  0  ->  memory[0] = 10
m[0, 1] = 20  ->  byte  4  ->  memory[1] = 20
m[0, 2] = 30  ->  byte  8  ->  memory[2] = 30
m[1, 0] = 40  ->  byte 12  ->  memory[3] = 40
m[1, 1] = 50  ->  byte 16  ->  memory[4] = 50
m[1, 2] = 60  ->  byte 20  ->  memory[5] = 60


#### Transposing = swapping the strides

`m.T` does not move any data. It looks at **the same memory** and just swaps the strides:

* in `m.T` the next **column** is the next row of `m` → **12 bytes**
* the next **row** is the next column of `m` → **4 bytes**

So `m.T.strides == (4, 12)`. That is why transposing a huge array is instant.

In [7]:
t = m.T

print("m.T:\n", t)
print("m.T.strides:", t.strides)
print("Same memory as m?", np.shares_memory(m, t))

# the same formula still works
i, j = 2, 1
print(f"m.T[{i}, {j}] =", t[i, j], " offset:", i * t.strides[0] + j * t.strides[1], "bytes")

m.T:
 [[10 40]
 [20 50]
 [30 60]]
m.T.strides: (4, 12)
Same memory as m? True
m.T[2, 1] = 60  offset: 20 bytes


#### General rule

For an array with any number of dimensions:

```
offset(a[i0, i1, i2, ...]) = i0 * strides[0] + i1 * strides[1] + i2 * strides[2] + ...
```

* the **last** stride is usually `itemsize` (next element in memory),
* every earlier stride = the next stride × the size of the next axis.

Now the same idea in 3D. Note that the array below has the default type `int64`, so each element takes **8 bytes**.

In [8]:
y = np.reshape(np.arange(2*3*4), (2,3,4))
y

array([[[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]],

       [[12, 13, 14, 15],
        [16, 17, 18, 19],
        [20, 21, 22, 23]]])

In [9]:
y.strides # shape (2,3,4)

(96, 32, 8)

In [10]:
z = y.T
z.strides

(8, 32, 96)

In [11]:
y[1,1,1]

np.int64(17)

At this step, we want to calculate how many bytes we need to "jump" in memory to access the element.

In [12]:
offset=sum(y.strides * np.array((1,1,1)))
offset/y.itemsize

# Simpler:
# i0, i1, i2 = 1, 1, 1
# offset_bytes = y.strides[0] * i0 + y.strides[1] * i1 + y.strides[2] * i2

np.float64(17.0)

What does `y.strides * np.array((1, 1, 1))` mean?

Each dimension of the array (2, 3, 4) has its own stride, which tells us how far apart elements are in memory:
* For the first dimension (axis 0), we need to jump 96 bytes to move to the next element.
* For the second dimension (axis 1), we jump 32 bytes.
* For the third dimension (axis 2), we jump 8 bytes.

By summing these values we obtain the offset.

What about `offset / y.itemsize`?

Each element occupies 8 bytes (y.itemsize == 8). To find the position of the element in the flattened 1D representation, we divide `offset / y.itemsize`.

## NumPy Sorting & Uniqueness Functions

### `np.sort()` – Sort Values

Returns a sorted copy of an array along a given axis.

* By default, sorts along the last axis (`axis = -1`).
* Does **not** modify the original array (use `.sort()` method for in-place sorting).
* Works on multidimensional arrays — you can choose which axis to sort.

#### Example – Sorting model predictions

In [13]:
import numpy as np

# Suppose we have predicted probabilities for 3 classes
probs = np.array([0.65, 0.15, 0.20])

print("Original probabilities:", probs)
print("Sorted probabilities:", np.sort(probs))

Original probabilities: [0.65 0.15 0.2 ]
Sorted probabilities: [0.15 0.2  0.65]


#### Example 2

In [14]:
import numpy as np

# Rows = samples, Columns = class probabilities
probs_2d = np.array([
    [0.65, 0.15, 0.20],
    [0.10, 0.80, 0.50],
    [0.30, 0.40, 0.90]
])

print("Original 2D array:\n", probs_2d)

# Sort without specifying axis (default: axis = -1 → sort within each row)
print("\nSorted along last axis (axis=-1):\n", np.sort(probs_2d))

# Sort along rows (axis=0) — sorts *down* each column
print("\nSorted along axis=0 (down each column):\n", np.sort(probs_2d, axis=0))

# Sort along columns (axis=1) — sorts *across* each row
print("\nSorted along axis=1 (across each row):\n", np.sort(probs_2d, axis=1))


Original 2D array:
 [[0.65 0.15 0.2 ]
 [0.1  0.8  0.5 ]
 [0.3  0.4  0.9 ]]

Sorted along last axis (axis=-1):
 [[0.15 0.2  0.65]
 [0.1  0.5  0.8 ]
 [0.3  0.4  0.9 ]]

Sorted along axis=0 (down each column):
 [[0.1  0.15 0.2 ]
 [0.3  0.4  0.5 ]
 [0.65 0.8  0.9 ]]

Sorted along axis=1 (across each row):
 [[0.15 0.2  0.65]
 [0.1  0.5  0.8 ]
 [0.3  0.4  0.9 ]]


### `np.argsort()` – Get Sorted Indices

Returns the **indices** that would sort the array.
This is useful when you want the *ranking* or to reorder another array accordingly.

#### Example – Ranking model predictions

In [15]:
import numpy as np

# Probabilities for 3 classes
probs = np.array([0.65, 0.15, 0.20])

sorted_idx = np.argsort(probs)
print("Sorted indices:", sorted_idx)
print("Classes from lowest to highest probability:", sorted_idx)
print("Classes from highest to lowest probability:", sorted_idx[::-1])

Sorted indices: [1 2 0]
Classes from lowest to highest probability: [1 2 0]
Classes from highest to lowest probability: [0 2 1]


###  `np.unique()` – Find Unique Values

Finds all unique values in an array (optionally, also their counts or indices).
Useful for identifying **unique categories**, **class labels**, or **distinct values** in datasets.

#### Example – Unique class labels in dataset

In [16]:
labels = np.array(["cat", "dog", "dog", "cat", "bird", "dog"])

unique_labels = np.unique(labels)
print("Unique class labels:", unique_labels)

Unique class labels: ['bird' 'cat' 'dog']


### Summary

* **`np.sort`** get ordered values
* **`np.argsort`** get order indices (for ranking)
* **`np.unique`** get unique or categorical values

## Random Data Generation in NumPy

The `numpy.random` module provides functions for generating random numbers.
These are widely used for:

* Sampling data
* Initializing model weights
* Random shuffling
* Data augmentation

### Understanding the Random Seed

Every time you call a random function, NumPy uses an internal algorithm to produce random-looking numbers.
However, these numbers are **not truly random** — they’re **pseudo-random**, generated by a deterministic process.

The **random seed** is the *starting point* for this process.
If you set the same seed, NumPy will always produce the same random numbers.

Setting `np.random.seed(n)` makes your random results **reproducible** — crucial for experiments and debugging in machine learning.

In [17]:
np.random.seed(42)
print("Run 1:", np.random.rand(3))

np.random.seed(42)
print("Run 2:", np.random.rand(3))

Run 1: [0.37454012 0.95071431 0.73199394]
Run 2: [0.37454012 0.95071431 0.73199394]


### Differences Between `rand`, `randint`, and `randn`

| Function                             | Description                                              | Output Range  |
| ------------------------------------ | -------------------------------------------------------- | ------------- |
| `np.random.rand(shape)`              | Uniform random floats                                    | [0, 1)        |
| `np.random.randint(low, high, size)` | Random integers                                          | [low, high) |
| `np.random.randn(shape)`             | Random floats from *normal distribution* (mean=0, std=1) | (-∞, +∞)      | 

In [18]:
np.random.seed(0)

print("rand:", np.random.rand(3))      # uniform
print("randint:", np.random.randint(0, 10, 3))  # integers
print("randint:", np.random.randint(0, 10, (3, 3)))  # integers
print("randn:", np.random.randn(3))    # normal distribution

rand: [0.5488135  0.71518937 0.60276338]
randint: [3 7 9]
randint: [[3 5 2]
 [4 7 6]
 [8 8 1]]
randn: [0.14404357 1.45427351 0.76103773]


# Advanced NumPy

## Broadcasting

The term **broadcasting** describes how NumPy treats arrays with different shapes during arithmetic operations. Subject to certain constraints, the smaller array is “broadcast” across the larger array so that they have compatible shapes.

NumPy operations are usually done on pairs of arrays on an element-by-element basis. In the simplest case, the two arrays must have exactly the same shape, as in the following example:

In [19]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([2.0, 2.0, 2.0])
a * b

array([2., 4., 6.])

NumPy’s broadcasting rule relaxes this constraint when the arrays’ shapes meet certain constraints. The simplest broadcasting example occurs when an array and a scalar value are combined in an operation:

In [20]:
a = np.array([1.0, 2.0, 3.0])
b = 2.0
a * b

array([2., 4., 6.])

The result matches the previous example where `b` was an array. Conceptually, `b` is stretched to match `a`'s shape, with copies of the scalar. However, NumPy optimizes this without creating actual copies, ensuring efficient memory and computation.

![Broadcasting](Images/broadcasting.png)

### General broadcasting rules

When operating on two arrays, NumPy compares their shapes element-wise. It starts with the trailing (i.e. rightmost) dimension and works its way left. Two dimensions are compatible when

1. they are equal, or
2. one of them is 1.

Input arrays can have different dimensions. The result matches the array with the most dimensions, using the largest size per dimension, assuming missing dimensions are size one.

### Example 1

Scaling RGB values for an image

In [21]:
image = np.random.rand(256, 256, 3)  # 256x256 image with 3 color channels
scale = np.array([0.5, 1.0, 1.5])  # Different scaling factors for each channel

# Broadcasting operation
scaled_image = image * scale

print("Image shape:", image.shape)
print("Scale shape:", scale.shape)
print("Result shape:", scaled_image.shape)

# Image shape:  (256, 256, 3)
# Scale shape:            (3,)
# Result shape: (256, 256, 3)

Image shape: (256, 256, 3)
Scale shape: (3,)
Result shape: (256, 256, 3)


### Example 2

Broadcasting with different shaped arrays

In [22]:
A = np.random.rand(8, 1, 6, 1)  # 4D array
B = np.random.rand(   7, 1, 5)  # 3D array

# Broadcasting operation
result = A * B

print("A shape:", A.shape)
print("B shape:", B.shape)
print("Result shape:", result.shape)

# A shape:      (8, 1, 6, 1)
# B shape:         (7, 1, 5)
# Result shape: (8, 7, 6, 5)

A shape: (8, 1, 6, 1)
B shape: (7, 1, 5)
Result shape: (8, 7, 6, 5)


### Example 3

Examples of arrays that can't be broadcast.

In [23]:
try:
    A = np.random.rand(3)
    B = np.random.rand(4)
    result = A + B 
except ValueError as e:
    print(e)

operands could not be broadcast together with shapes (3,) (4,) 


In [24]:
try:
    A = np.random.rand(   2, 1)
    B = np.random.rand(8, 4, 3)
    result = A + B 
except ValueError as e:
    print(e)

operands could not be broadcast together with shapes (2,1) (8,4,3) 


### Example 4

Broadcasting when a 1-d array is added to a 2-d array

In [25]:
A = np.array([[ 0.0,  0.0,  0.0],
              [10.0, 10.0, 10.0],
              [20.0, 20.0, 20.0],
              [30.0, 30.0, 30.0]])
B = np.array([1.0, 2.0, 3.0])
A + B

array([[ 1.,  2.,  3.],
       [11., 12., 13.],
       [21., 22., 23.],
       [31., 32., 33.]])

![Broadcastin1D2D](Images/broadcasting_2.png)

### Example 5

Broadcasting provides a convenient way of taking the outer product (or any other outer operation) of two arrays. The following example shows an outer addition operation of two 1-d arrays

In [26]:
a = np.array([0.0, 10.0, 20.0, 30.0])
b = np.array([1.0, 2.0, 3.0])
a[:, np.newaxis] + b

array([[ 1.,  2.,  3.],
       [11., 12., 13.],
       [21., 22., 23.],
       [31., 32., 33.]])

![Broadcasting](Images/broadcasting_4.png)

Here the newaxis index operator inserts a new axis into a, making it a two-dimensional 4x1 array. Combining the 4x1 array with b, which has shape (3,), yields a 4x3 array.

## Structured arrays

Structured arrays are ndarrays whose datatype is a composition of simpler datatypes organized as a sequence of named fields. For example:

In [27]:
dt = np.dtype([('time', [('min', np.int64), ('sec', np.int64)]),
               ('temp', float)])
x = np.array([((10, 0), 2), ((10,30), 3), ((11,00), 5)], 
             dtype=dt)
x

array([((10,  0), 2.), ((10, 30), 3.), ((11,  0), 5.)],
      dtype=[('time', [('min', '<i8'), ('sec', '<i8')]), ('temp', '<f8')])

In [28]:
print(f'x[time]: {x["time"]}')

x[time]: [(10,  0) (10, 30) (11,  0)]


In [29]:
print(f'x[temp]: {x["temp"]}')

x[temp]: [2. 3. 5.]


## Masked Arrays in NumPy

### Why Masked Arrays?

In real-world datasets, some values can be **missing** or **invalid**.
For example:

* A temperature sensor may fail and return `-999`.
* A survey might have unanswered fields.
* Some data could be corrupted or physically impossible.

The **`numpy.ma`** module helps handle such cases without removing or replacing values manually.
It lets you **“mask”** bad data — i.e., mark it as invalid so NumPy ignores it in computations.

### Basic Concept

A **masked array** combines:

* A **data array** (like any NumPy `ndarray`)
* A **mask**: Boolean array where

  * `True` → value is invalid (masked)
  * `False` → value is valid (used in computations)

Masked values are automatically excluded from calculations like `mean()`, `sum()`, `max()`, etc.

### Example 1

In [30]:
import numpy.ma as ma

# Example dataset: sensor readings
x = np.array([1, 2, 3, -1, 5])
print("Raw sensor readings:", x)

# Suppose -1 means an invalid measurement
mask = [0, 0, 0, 1, 0]  # mask=True (1) marks invalid entries

mx = ma.masked_array(x, mask=mask)
print("\nMasked array:\n", mx)
print("Mask:\n", mx.mask)

# Compute mean while ignoring invalid (masked) values
print("\nMean (ignoring masked values):", mx.mean())
print("Sum (ignoring masked values):", mx.sum())
print("Standard deviation (ignoring masked values):", mx.std())

Raw sensor readings: [ 1  2  3 -1  5]

Masked array:
 [1 2 3 -- 5]
Mask:
 [False False False  True False]

Mean (ignoring masked values): 2.75
Sum (ignoring masked values): 11
Standard deviation (ignoring masked values): 1.479019945774904


### Example 2 – Replacing Masked Values

You can “fill in” masked values with a constant (e.g., 0 or mean).

In [31]:
temps = np.array([20, 21, -999, 23, 22])
mtemps = ma.masked_equal(temps, -999)

print("Masked array:", mtemps)
print("Filled with 0:", mtemps.filled(0))
print("Filled with mean value:", mtemps.filled(mtemps.mean()))


Masked array: [20 21 -- 23 22]
Filled with 0: [20 21  0 23 22]
Filled with mean value: [20 21 21 23 22]


### Summary

| Function                    | Description                                 |
| --------------------------- | ------------------------------------------- |
| `ma.masked_equal(x, value)` | Mask all entries equal to a given value     |
| `ma.masked_where(cond, x)`  | Mask where condition is True                |
| `ma.masked_invalid(x)`      | Mask NaN or inf values                      |
| `ma.mean(x)`                | Mean ignoring masked values                 |
| `ma.sum(x)`                 | Sum ignoring masked values                  |
| `ma.filled(x, fill_value)`  | Replace masked values with given fill value |

## Stride Tricks

In [32]:
from numpy.lib import stride_tricks

### Reminder: Strides

#### What Are Strides?

In NumPy, **strides** show how many **bytes in memory** you need to move to reach the next element **along each axis**.

#### Example – 2D Array

In [33]:
x = np.array([[1, 2, 3],
              [4, 5, 6]])
print("Array:\n", x)
print("Shape:", x.shape)
print("Item size (bytes):", x.itemsize)
print("Strides:", x.strides)

Array:
 [[1 2 3]
 [4 5 6]]
Shape: (2, 3)
Item size (bytes): 8
Strides: (24, 8)


#### Interpretation

Each element takes **8 bytes**.

* The second stride (8) → to move from one element to the next **in the same row**,
  NumPy jumps **8 bytes** in memory.
* The first stride (24) → to move to the next **row**, NumPy skips **24 bytes**
  (3 columns × 8 bytes = 24 bytes).

### numpy.lib.stride_tricks.sliding_window_view

The `numpy.lib.stride_tricks.sliding_window_view()` function creates sliding window views of N-dimensional arrays without copying data. This function is safer and easier to use than `as_strided()` for tasks like generating moving windows for convolution or analysis.

**Function Parameters**

| Parameter        | Description                                                  |
|------------------|--------------------------------------------------------------|
| `x`              | Input NumPy array                                            |
| `window_shape`   | Size of the sliding window (int or tuple for N-dim arrays)   |
| `axis` (optional) | Axis or axes to slide over; defaults to all axes            |

**Returns**

`view : ndarray`  
A new view of the input array with overlapping sliding windows.


#### Example 1D

In [34]:
x = np.arange(10)
print("x:", x)

windows = stride_tricks.sliding_window_view(x, window_shape=3)
print("Sliding window:\n", windows)

x: [0 1 2 3 4 5 6 7 8 9]
Sliding window:
 [[0 1 2]
 [1 2 3]
 [2 3 4]
 [3 4 5]
 [4 5 6]
 [5 6 7]
 [6 7 8]
 [7 8 9]]


#### Example 2D

In [35]:
import numpy as np
from numpy.lib import stride_tricks

# Create a 2D array
A = np.arange(1, 17).reshape(4, 4)
print("A:\n", A)

# Create 2x2 windows
windows = stride_tricks.sliding_window_view(A, window_shape=(2, 2))
print("\nSliding 2x2 windows:\n", windows)

print("\nShape of windows:", windows.shape)


A:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]
 [13 14 15 16]]

Sliding 2x2 windows:
 [[[[ 1  2]
   [ 5  6]]

  [[ 2  3]
   [ 6  7]]

  [[ 3  4]
   [ 7  8]]]


 [[[ 5  6]
   [ 9 10]]

  [[ 6  7]
   [10 11]]

  [[ 7  8]
   [11 12]]]


 [[[ 9 10]
   [13 14]]

  [[10 11]
   [14 15]]

  [[11 12]
   [15 16]]]]

Shape of windows: (3, 3, 2, 2)


For a 2D array `A` of shape (4,4), `sliding_window_view(A, (2,2))` produces a 4-D view of shape (3,3,2,2).  
The first two dimensions (3,3) represent the possible window positions,  
and the last two (2,2) are the shape of each window.  
Each element `windows[i,j]` is a 2×2 subarray `A[i:i+2, j:j+2]`.  
This is useful for vectorized convolution, pooling, or local neighborhood operations — all done without copying data.

### numpy.lib.stride_tricks.as_strided

The `numpy.lib.stride_tricks.as_strided()` function allows you to create custom views of NumPy arrays without copying data. It works by manipulating memory strides, enabling you to define a custom way of interpreting the underlying data.

**Function Parameters**

| Parameter     | Description                                           |
|---------------|-------------------------------------------------------|
| `x`           | Input NumPy array                                     |
| `shape`       | Desired shape of the new view                         |
| `strides`     | Bytes to move between elements in each dimension      |
| `subok`       | Keeps subclass type if `True` (default)               |
| `writeable`   | If `True`, allows changes to the view (use cautiously)|

**Returns**

`view : ndarray`
A new view of the input array with overlapping sliding windows.


**Risks**
1. **Memory violation risk** Manipulating strides allows reading data outside the valid array range, which can cause errors or undefined behavior.
2. **No data copying** Modifying elements of a view created with as_strided will also modify the original data.

#### Example
It can be used to efficiently perform operations on input data, such as creating sliding windows without memory overhead. This is particularly useful for convolutional operations.

In [36]:
x = np.arange(10)

window_size = 3
stride = x.strides[0]

windows = stride_tricks.as_strided(x, 
                                   shape=(len(x) - window_size + 1, window_size), 
                                   strides=(stride, stride))
print(windows)

[[0 1 2]
 [1 2 3]
 [2 3 4]
 [3 4 5]
 [4 5 6]
 [5 6 7]
 [6 7 8]
 [7 8 9]]


**How It Works**

1. Create the input array x
2. Define the window size and stride
    * window_size = 3: We want overlapping sliding windows of length 3.
    * x.strides[0]: The stride for a 1D NumPy array is the number of bytes required to move from one element to the next.
3. Set the shape of the new view `shape=(len(x) - window_size + 1, window_size)`
    * len(x) = 10
    * We want windows of size 3, so the number of possible windows is 10 - 3 + 1 = 8. This makes the shape of the resulting array (8, 3).
4. Set the strides `strides=(stride, stride)`
    * The first value tells NumPy how many bytes to move when selecting the starting point of the next window (row).
    * The second value tells NumPy how many bytes to move between elements within a window (column).
